# Lesson 2: Binary Search Trees, Traversals, and Balancing

**Source Reference:** [Data Structures and Algorithms in Python (freeCodeCamp / Jovian)](https://www.youtube.com/watch?v=pkYVOmU3MgA)

## Problem 1: Fast In-Memory User Database
> **Question:** As a senior backend engineer, you are tasked with developing a fast in-memory data structure to manage profile information (username, name, and email) for 100 million users. It should allow the following operations to be performed efficiently: Insert, Find, Update, and List all users sorted by username.

### Input and Output Formats
* **Input:** User profiles containing a unique `username`, `name`, and `email`.
* **Output:** A unified data structure executing insertion, retrieval, modification, and ordered listing.

In [1]:
class User:
    """Standard entity class representing a user profile."""
    def __init__(self, username, name, email):
        self.username = username
        self.name = name
        self.email = email
        
    def __repr__(self):
        # Provides a clean, developer-friendly string representation in the console
        return f"User(username='{self.username}', name='{self.name}', email='{self.email}')"

# Test Suite: Sample users to simulate database operations
aakash = User('aakash', 'Aakash Rai', 'aakash@example.com')
biraj = User('biraj', 'Biraj Das', 'biraj@example.com')
hemanth = User('hemanth', 'Hemanth Jain', 'hemanth@example.com')
jadhesh = User('jadhesh', 'Jadhesh Verma', 'jadhesh@example.com')
siddhant = User('siddhant', 'Siddhant Sinha', 'siddhant@example.com')
sonaksh = User('sonaksh', 'Sonaksh Kumar', 'sonaksh@example.com')
vishal = User('vishal', 'Vishal Goel', 'vishal@example.com')

users = [aakash, biraj, hemanth, jadhesh, siddhant, sonaksh, vishal]

## Approach 1: Sorted List (Brute Force)
**Strategy:** Store `User` objects in a standard Python list, maintaining alphabetical order by `username`. To insert, find, or update, iterate through the list sequentially.

### Complexity Analysis
* **Time Complexity:** $O(N)$ for Insert, Find, and Update. Iterating through 100 million users to find a target or shift elements during insertion takes ~10 seconds per request, which will crash a production server.
* **Space Complexity:** $O(1)$ additional space beyond the stored list data.

In [2]:
class UserDatabaseLinear:
    def __init__(self):
        self.users = []
    
    def insert(self, user):
        i = 0
        # Linear scan to find the correct alphabetical position
        while i < len(self.users):
            if self.users[i].username > user.username:
                break
            i += 1
        # Inserting into a python list shifts all subsequent elements O(N)
        self.users.insert(i, user)
    
    def find(self, username):
        # Linear search for the user O(N)
        for user in self.users:
            if user.username == username:
                return user
        return None
    
    def update(self, user):
        # Relies on find() to locate, then mutates the object
        target = self.find(user.username)
        if target is not None:
            target.name, target.email = user.name, user.email
        
    def list_all(self):
        return self.users

## Approach 2: Binary Search Tree (Optimized)
We can optimize this by organizing the data hierarchically into a **Binary Tree**. To make searching efficient, we enforce the **Binary Search Tree (BST) Property**: 
1. The **left subtree** of any node contains only keys strictly *smaller* than the node's key.
2. The **right subtree** of any node contains only keys strictly *greater* than the node's key.

### Complexity Analysis
By branching left or right, we halve the search space at each step.
* **Time Complexity:** $O(\log N)$ for Insert, Find, and Update. Checking 100 million records takes roughly 26 operations instead of 100,000,000. Listing all takes $O(N)$ via an Inorder Traversal.
* **Space Complexity:** $O(h)$ where $h$ is the height of the tree (due to the recursive call stack).

In [3]:
class BSTNode:
    """A binary tree node mapping a key to a payload and left/right branches."""
    def __init__(self, key, value=None):
        self.key = key
        self.value = value
        self.left = None
        self.right = None
        self.parent = None # Helpful for upward traversal if needed

def insert(node, key, value):
    """Recursively traverses the tree to attach a new key-value pair."""
    # Base case: Found an empty spot, insert new node here
    if node is None:
        return BSTNode(key, value)
        
    # If the key is smaller, traverse down the left branch
    elif key < node.key:
        node.left = insert(node.left, key, value)
        node.left.parent = node
        
    # If the key is larger, traverse down the right branch
    elif key > node.key:
        node.right = insert(node.right, key, value)
        node.right.parent = node
        
    return node

def find(node, key):
    """Recursively traces the BST bounds to locate a key in O(log N) time."""
    if node is None:
        return None
    if key == node.key:
        return node
        
    # Apply BST property to halve the search space
    if key < node.key:
        return find(node.left, key)
    if key > node.key:
        return find(node.right, key)

def update(node, key, value):
    """Locates a node using find() and safely mutates its payload."""
    target = find(node, key)
    if target is not None:
        target.value = value

def list_all(node):
    """
    Inorder traversal: Left Subtree -> Current Node -> Right Subtree.
    For a BST, this guarantees elements are returned in strict sorted order. O(N).
    """
    if node is None:
        return []
    return list_all(node.left) + [(node.key, node.value)] + list_all(node.right)

## Problem 2: Validating and Balancing a BST
> **Question:** If elements are inserted into a BST in already-sorted order, the tree becomes skewed (like a single-sided linked list), degrading operations back to $O(N)$. Write functions to check if a tree is balanced, and to rebuild it perfectly balanced.

**Strategy:** A tree is balanced if the height of the left and right subtrees of *any* node differ by no more than 1. 
To balance a skewed tree:
1. Perform an `inorder` traversal to extract all nodes into a sorted array. 
2. Recursively pick the exact middle element of the array to act as the root of each new subtree.

### Complexity Analysis for Balancing
* **Time Complexity:** $O(N)$ to traverse the tree, plus $O(N)$ to recursively build the new balanced tree.
* **Space Complexity:** $O(N)$ to store the array of extracted nodes in memory.

In [4]:
def is_balanced(node):
    """Recursively verifies height differentials between branches do not exceed 1."""
    # Base case: An empty node is perfectly balanced with height 0
    if node is None:
        return True, 0
    
    # Recursively check left and right branches
    balanced_l, height_l = is_balanced(node.left)
    balanced_r, height_r = is_balanced(node.right)
    
    # Tree is balanced if both branches are balanced and heights differ by <= 1
    balanced = balanced_l and balanced_r and abs(height_l - height_r) <= 1
    height = 1 + max(height_l, height_r)
    
    return balanced, height

def make_balanced_bst(data, lo=0, hi=None, parent=None):
    """Rebuilds a tree from a sorted array by forcing the midpoint to the root."""
    if hi is None:
        hi = len(data) - 1
    # Base condition to stop recursion
    if lo > hi:
        return None
    
    # Find the absolute middle element to serve as the root
    mid = (lo + hi) // 2
    key, value = data[mid]

    root = BSTNode(key, value)
    root.parent = parent
    
    # Recursively attach the bisected left and right segments
    root.left = make_balanced_bst(data, lo, mid - 1, root)
    root.right = make_balanced_bst(data, mid + 1, hi, root)
    
    return root

def balance_bst(node):
    """Wrapper function to flatten an unbalanced tree and rebuild it."""
    # list_all performs an inorder traversal, returning sorted data
    return make_balanced_bst(list_all(node))

## The Production Abstraction: TreeMap
A backend engineer does not expose raw tree nodes to the end user. We wrap the complex recursive logic inside a Python-friendly class using magic methods to emulate a standard Python Dictionary interface.

### Final Complexity Engine
* **Insert / Update (`treemap['a'] = x`)**: $O(\log N)$
* **Find (`x = treemap['a']`)**: $O(\log N)$
* **Size Check (`len(treemap)`)**: $O(N)$ *(Can be optimized to $O(1)$ by caching size)*
* **Iteration (`for k, v in treemap`)**: $O(N)$

In [5]:
class TreeMap():
    """A self-contained Dictionary-like interface powered by a Balanced BST."""
    def __init__(self):
        self.root = None
        
    def __setitem__(self, key, value):
        """Overrides `treemap[key] = value`. Handles both insert and update."""
        node = find(self.root, key)
        if not node:
            self.root = insert(self.root, key, value)
            # Force rebalance on new insertions to prevent skewed O(N) degradation.
            # (Note: In production AVL/Red-Black trees, balancing is O(1) via localized rotations)
            self.root = balance_bst(self.root)
        else:
            update(self.root, key, value)
            
    def __getitem__(self, key):
        """Overrides `value = treemap[key]`."""
        node = find(self.root, key)
        return node.value if node is not None else None
    
    def __iter__(self):
        """Allows direct Python looping: `for item in treemap`."""
        return (x for x in list_all(self.root))
    
    def __len__(self):
        """Overrides `len(treemap)` by counting nodes recursively."""
        def tree_size(node):
            if node is None: return 0
            return 1 + tree_size(node.left) + tree_size(node.right)
        return tree_size(self.root)